## Semantic Query Caching Simulation — DBpedia (`tab:caching`)

Implements the protocol in `notebooks/caching/README.md` exactly as committed
to in the paper (§Downstream Application):

- **Stream:** held-out SQuARE **test split** anchors, seeded shuffle.
- **Cache:** FIFO over the **K most recent** executed queries (embedding +
  result set). Hit when max cosine ≥ θ → return the cached result, do not
  insert. Miss → "execute" (oracle lookup) and insert.
- **Grid:** K ∈ {50, 100, 200} × θ ∈ {0.7, 0.8, 0.9}.
- **Models:** SparqlCE, TF-IDF, E5-base-v2, Qwen3-Embedding-0.6B.
- **Metrics:** hit rate; result quality = mean answer-set Jaccard between the
  returned (cached) result and the true result, over hits.

Requires the SparqlCE checkpoint `sparqlce/models/final_model_mrr_0.4487.pt`
(not in git — see `notebooks/visualization/FIGURE3_RUNBOOK.md`). Everything
else is in `datasets/`. Outputs `results/caching/caching_dbpedia.json` and
ready-to-paste LaTeX rows for `tab:caching`.


In [ ]:
import os
import sys
import json
import random
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# ---- Project root (works from notebooks/caching/ or anywhere in the repo) ----
_PROJECT_ROOT = Path.cwd()
while not (_PROJECT_ROOT / "datasets").exists():
    assert _PROJECT_ROOT != _PROJECT_ROOT.parent, "could not locate repo root"
    _PROJECT_ROOT = _PROJECT_ROOT.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

# ---- Config ----
TRIPLETS_PATH   = _PROJECT_ROOT / "datasets" / "dbpedia-triplets-dataset.json"
SPLITS_PATH     = _PROJECT_ROOT / "datasets" / "splits_dbpedia.json"
RESULTS_PATH    = _PROJECT_ROOT / "datasets" / "result_sets_dbpedia.json"
SPARQLCE_CKPT   = _PROJECT_ROOT / "sparqlce" / "models" / "final_model_mrr_0.4487.pt"
OUT_DIR         = _PROJECT_ROOT / "results" / "caching"

SEED       = 42
K_GRID     = [50, 100, 200]
THETA_GRID = [0.7, 0.8, 0.9]
HEADLINE   = (100, 0.8)          # (K, theta) row reported in the main table

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"project root: {_PROJECT_ROOT}\ndevice: {DEVICE}")


In [ ]:
# ============================================================================
# 1. Build the query stream: test-split anchors with a non-empty oracle result
# ============================================================================
with open(TRIPLETS_PATH, encoding="utf-8") as f:
    triplets = json.load(f)
with open(SPLITS_PATH) as f:
    splits = json.load(f)
with open(RESULTS_PATH, encoding="utf-8") as f:
    result_sets = {q: frozenset(rows) for q, rows in json.load(f).items() if rows}

test_anchors  = [triplets[i]["anchor_query"] for i in splits["test_idx"]]
train_anchors = [triplets[i]["anchor_query"] for i in splits["train_idx"]]

stream  = [q for q in test_anchors if q in result_sets]
dropped = len(test_anchors) - len(stream)

random.seed(SEED)
random.shuffle(stream)

print(f"test anchors: {len(test_anchors)} | in stream: {len(stream)} "
      f"| dropped (missing/empty oracle result): {dropped}")


def jaccard(a: frozenset, b: frozenset) -> float:
    union = len(a | b)
    return len(a & b) / union if union else 0.0


In [ ]:
# ============================================================================
# 2. Embed the stream once per model (all embeddings L2-normalised)
# ============================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

stream_embeddings = {}

# --- SparqlCE (trained checkpoint) ---
from sparqlce.embedder import SPARQLEmbedder
sparqlce = SPARQLEmbedder(str(SPARQLCE_CKPT))
embs = []
for i in tqdm(range(0, len(stream), 32), desc="SparqlCE"):
    embs.append(sparqlce.encode(stream[i:i + 32]))
stream_embeddings["SparqlCE"] = normalize(np.vstack(embs))

# --- TF-IDF (fitted on the TRAIN split only; sklearn norm='l2' by default) ---
tfidf = TfidfVectorizer().fit(train_anchors)
stream_embeddings["TF-IDF"] = np.asarray(tfidf.transform(stream).todense())

# --- E5-base-v2 (zero-shot, "query: " prefix per model card) ---
e5 = SentenceTransformer("intfloat/e5-base-v2", device=DEVICE)
stream_embeddings["E5-base-v2"] = e5.encode(
    [f"query: {q}" for q in stream],
    convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
del e5

# --- Qwen3-Embedding-0.6B (zero-shot) ---
qwen = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=DEVICE)
stream_embeddings["Qwen3-Embedding"] = qwen.encode(
    stream, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True, batch_size=8)
del qwen
if torch.cuda.is_available():
    torch.cuda.empty_cache()

for name, E in stream_embeddings.items():
    print(f"{name:>16}: {E.shape}")


In [ ]:
# ============================================================================
# 3. Cache simulation (spec: FIFO, hit => return cached result, no insert)
# ============================================================================
def simulate(stream, E, result_sets, K, theta):
    """One pass over the stream. E[i] is the embedding of stream[i]."""
    cache_emb, cache_res = deque(maxlen=K), deque(maxlen=K)
    hits, qualities = 0, []
    for i, q in enumerate(stream):
        R_true = result_sets[q]
        if cache_emb:
            sims = np.stack(cache_emb) @ E[i]
            best = int(np.argmax(sims))
            if sims[best] >= theta:
                hits += 1
                qualities.append(jaccard(cache_res[best], R_true))
                continue                       # hit: served from cache
        cache_emb.append(E[i])                 # miss: "execute" + insert
        cache_res.append(R_true)
    return {
        "hit_rate": hits / len(stream),
        "result_quality": float(np.mean(qualities)) if qualities else 0.0,
        "n_hits": hits,
    }


all_results = {}
for name, E in stream_embeddings.items():
    all_results[name] = {}
    for K in K_GRID:
        for theta in THETA_GRID:
            r = simulate(stream, E, result_sets, K, theta)
            all_results[name][f"K={K},theta={theta}"] = r
    print(f"done: {name}")


In [ ]:
# ============================================================================
# 4. Save the full grid + display
# ============================================================================
OUT_DIR.mkdir(parents=True, exist_ok=True)
payload = {
    "kg": "dbpedia",
    "seed": SEED,
    "stream_length": len(stream),
    "dropped": dropped,
    "grid": {"K": K_GRID, "theta": THETA_GRID},
    "results": all_results,
}
out_json = OUT_DIR / "caching_dbpedia.json"
with open(out_json, "w") as f:
    json.dump(payload, f, indent=2)
print(f"saved {out_json}\n")

rows = []
for name, grid in all_results.items():
    for cfg, r in grid.items():
        K, theta = cfg.replace("K=", "").split(",theta=")
        rows.append({"model": name, "K": int(K), "theta": float(theta),
                     "hit_rate": r["hit_rate"], "result_quality": r["result_quality"]})
df = pd.DataFrame(rows)

print("Hit rate:")
print(df.pivot_table(index="model", columns=["K", "theta"], values="hit_rate").round(3))
print("\nResult quality (mean Jaccard over hits):")
print(df.pivot_table(index="model", columns=["K", "theta"], values="result_quality").round(3))


In [ ]:
# ============================================================================
# 5. LaTeX rows for tab:caching
# ============================================================================
K_h, theta_h = HEADLINE
order = ["SparqlCE", "TF-IDF", "E5-base-v2", "Qwen3-Embedding"]
latex_names = {"SparqlCE": r"\textsc{SparqlCE}", "TF-IDF": "TF-IDF",
               "E5-base-v2": "E5-base-v2", "Qwen3-Embedding": "Qwen3-Embedding"}

print(f"% tab:caching — headline operating point (K={K_h}, theta={theta_h}), "
      f"full grid in results/caching/caching_dbpedia.json")
for name in order:
    r = all_results[name][f"K={K_h},theta={theta_h}"]
    print(f"{latex_names[name]:<20} & $({K_h}, {theta_h})$ & "
          f"{r['hit_rate']:.3f} & {r['result_quality']:.3f} \\\\")

best = all_results["SparqlCE"][f"K={K_h},theta={theta_h}"]
alts = {n: all_results[n][f"K={K_h},theta={theta_h}"] for n in order[1:]}
best_base = max(alts, key=lambda n: alts[n]["result_quality"])
print(f"\n% headline sentence draft:")
print(f"% At K={K_h}, theta={theta_h}, SparqlCE serves {best['hit_rate']:.0%} of the stream "
      f"from cache with {best['result_quality']:.2f} result quality, vs. "
      f"{alts[best_base]['result_quality']:.2f} for the best text baseline ({best_base}).")


### Notes

- The oracle (`result_sets_dbpedia.json`) replaces live endpoint execution, so
  the simulation is deterministic and offline.
- Rerunning with a different stream order: change `SEED` — report the default
  seed-42 numbers in the paper.
- Wikidata twin requires `datasets/result_sets_wikidata.json` (not in the
  repo yet).
